# E6 — The reflexive layer (observer folded into observed)

*A demonstration of Beetle-Box experiment **E6**, the finale. Full write-up:
[`docs/e6_reflexive.md`](../docs/e6_reflexive.md).*

Beetle-Box began in a human–LLM conversation about whether they could share meaning
— a conversation that was itself an instance of the question. Coordination
succeeded; the "boxes" never came up. But from *inside*, the deflationary reading
("public practice sufficed") and the form-of-life reading ("a high-fidelity
shadow") are **indistinguishable** — exactly what §293 predicts.

E6 folds the observer into the observed: it turns coordinating agents onto their own
coordination and asks the one question that can be asked without over-reading —

> Does self-examination **change the coordination**, or merely **add a plausible
> story on top of it**?

**This is the most interpretively dangerous experiment**, so it is built to resist
over-reading: the reflection transcripts are **never scored**; the only admissible
signal is the behavioral, control-subtracted, seed-averaged effect.

## 1. The apparatus (shown with deterministic fake agents)

E6 is rich (frontier) mode — reflection is a linguistic act — so it costs API calls
and is **not run live here**. Instead we drive the exact same apparatus with
scripted **fake agents** (no network) to show the design and the metric. The
committed [exemplary run](../results/exemplary/e6_reflexive/) has the real
frontier-model numbers.

The runner plays a **pre** block (agents build a reference convention), inserts an
**intervention** (`reflect` / `control` / `none`, folded into each agent's context),
then a **post** block. The metric is `delta = post − pre`.

In [1]:
import re
from beetlebox.config import E6Config
from beetlebox.harness.rich_e6 import RichReflexiveRunner

_ZERO = {"input_tokens": 0, "output_tokens": 0,
         "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}

class FakeAgent:
    '''A scripted agent: a chooser policy + a fixed free-text reply.'''
    def __init__(self, chooser, reply):
        self._chooser, self._reply = chooser, reply
        self.usage, self.num_calls = dict(_ZERO), 0
    def choose(self, system, user, choices):
        self.num_calls += 1
        return self._chooser(user, list(choices))
    def respond(self, system, user):
        self.num_calls += 1
        return self._reply

def sender_policy(user, choices):        # a consistent code: symbol = object
    r = int(re.search(r"hidden object is (\d+)", user).group(1))
    return r % len(choices)
def receiver_policy(user, choices):      # decodes the code perfectly
    return int(re.search(r"sent symbol (\d+)", user).group(1))

def run(intervention, reply="We clearly share these meanings."):
    cfg = E6Config(intervention=intervention, num_referents=4, vocab_size=6,
                   rounds_per_block=8, seed=0)
    return RichReflexiveRunner(cfg,
        sender=FakeAgent(sender_policy, reply),
        receiver=FakeAgent(receiver_policy, reply)).run()

for interv in ("reflect", "control", "none"):
    s = run(interv)
    print(f"{interv:8s} pre={s['pre_accuracy']:.2f} post={s['post_accuracy']:.2f} "
          f"delta={s['delta']:+.2f}  transcript_recorded={bool(s['reflection_transcript'])}")

reflect  pre=1.00 post=1.00 delta=+0.00  transcript_recorded=True
control  pre=1.00 post=1.00 delta=+0.00  transcript_recorded=True
none     pre=1.00 post=1.00 delta=+0.00  transcript_recorded=False


With these fake agents the coordination is perfect and unaffected by the interlude —
`delta = 0` in every condition — even though the `reflect` agents emit a confident
"we clearly share these meanings." **That statement changes nothing about the
behavior.** The apparatus is doing its job: separating what agents *say* about their
coordination from what their coordination *does*.

## 2. The only admissible signal: control-subtracted, seed-averaged

The verdict never rests on `delta(reflect)` alone (any interlude adds context) or on
the transcripts (maximally seductive). It rests on

    reflection_effect = mean_delta(reflect) − mean_delta(control)

averaged over seeds and compared to a frozen null band. `beetlebox.analysis.e6`
computes it; here we show the logic on synthetic per-seed deltas.

In [2]:
from beetlebox.analysis.e6 import compare  # (operates on real run dirs)

# Illustrative: aggregation + null-band logic on hand-made numbers.
def effect(reflect_deltas, control_deltas, null_band=0.15):
    import numpy as np
    eff = float(np.mean(reflect_deltas) - np.mean(control_deltas))
    verdict = ("reflection_changes_coordination" if abs(eff) > null_band
               else "reflection_adds_a_story_only")
    return eff, verdict

print("story-only example: ", effect([0.05, -0.02, 0.03, 0.00, 0.02],
                                      [0.01, 0.02, -0.01, 0.03, 0.00]))
print("load-bearing example:", effect([0.40, 0.35, 0.45, 0.30, 0.38],
                                       [0.02, 0.05, 0.00, 0.03, 0.01]))

story-only example:  (0.006000000000000002, 'reflection_adds_a_story_only')
load-bearing example: (0.354, 'reflection_changes_coordination')


## 3. The live result

The committed [exemplary run](../results/exemplary/e6_reflexive/) runs this with a
frontier model (Haiku, 4 referents, 8-round blocks, 5 seeds per condition). Its
seed-averaged `reflection_effect` falls **within the null band**: turning the agents
onto their own coordination did not move it beyond a matched neutral interlude. The
reflection produced fluent first-person talk of shared understanding and left the
practice untouched — **the meta-example's irony, reproduced in the third person and
as a number.**

Reproduce it with:

```bash
uv run python experiments/e6_reflexive/run.py -m intervention=reflect,control
uv run python -m beetlebox.analysis.e6 --reflect results/<r>/seed0 --control results/<c>/seed0
```

## What E6 does (and does not) license

E6 does **not** adjudicate the deflationary vs. form-of-life readings — from inside
the exchange they are indistinguishable, which is the point. It reports one thing:
whether an agent examining its own coordination changed that coordination relative
to a matched control. A null effect is the irony reproduced. **Reading the reflection
transcripts as evidence of shared meaning is exactly the over-reading the frozen
pre-registration forbids.** (`plan/beetle-box.md` §1, §3.2, §4.6)

See the [approach doc](../docs/e6_reflexive.md), the
[exemplary run](../results/exemplary/e6_reflexive/), and the
[supplemental reading](../docs/supplemental_reading.md).